# Ddri Station-Day 데이터셋 설명 노트북

## 목적

- 예측용 `station-day` 데이터셋이 어떤 기준으로 생성됐는지 설명한다.
- target, 공휴일, 날씨, 군집 label, 운영 보조 지표가 어떻게 결합됐는지 정리한다.
- 읽는 사람이 최종 학습용 CSV 구조를 빠르게 이해할 수 있게 한다.

## 기준 정의

- target: `rental_count`
- grain: `station_id + date`
- 학습 구간: 2023~2024
- 테스트 구간: 2025

## 현재 포함된 주요 feature

- 시간/캘린더 정보
- 날씨 정보
- 군집 label
- 대여량, 반납량, self-return, 순유출입

## 해석 시 주의점

- self-return은 제거 대상이 아니라 운영 해석용 지표다.
- 신규 스테이션은 메인 평가셋에서 분리한다.
- 상업/업무 POI는 아직 보류 상태다.


## 1. 기본 경로와 데이터셋 로드

이 단계에서는 학습/테스트 베이스라인 데이터셋과 flow metric 요약 파일을 불러온다.

In [ ]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path('/Users/cheng80/Desktop/ddri_work')
PRED_DIR = BASE_DIR / 'works' / '03_prediction'
DATA_DIR = PRED_DIR / '02_data'

train_path = DATA_DIR / 'ddri_station_day_train_baseline_dataset.csv'
test_path = DATA_DIR / 'ddri_station_day_test_baseline_dataset.csv'
flow_summary_path = DATA_DIR / 'ddri_station_day_flow_metrics_summary.csv'

train_path, test_path

## 2. 컬럼 구조 확인

최종 CSV에는 target, 날씨, 공휴일, 군집 label, 운영 보조 지표가 함께 들어 있다.

In [ ]:
train_df = pd.read_csv(train_path, nrows=5)
train_df.columns.tolist()

## 3. 운영 보조 지표 확인

여기서 확인할 핵심 지표는 아래 네 가지다.

- `return_count`
- `same_station_return_count`
- `same_station_return_ratio`
- `net_flow`

이 지표들은 대여 수요와 재고 변동을 분리해서 해석하기 위한 목적이다.

In [ ]:
flow_summary = pd.read_csv(flow_summary_path)
flow_summary

## 4. 왜 self-return을 제거하지 않았는가

같은 대여소에서 빌리고 같은 대여소로 반납한 자전거는 하루 말 재고 변화는 작을 수 있다. 하지만 실제로 한 번 대여되었고, 운행 중에는 다른 이용자가 바로 쓸 수 없었다.

따라서 self-return은 이상치가 아니라 다음을 설명하는 보조 지표다.

- 실제 이용 수요
- 하루 말 재고 변화와의 차이
- 적정 보유 대수 해석 시 필요한 운영 정보


## 5. 다음 단계

이 데이터셋을 바탕으로 다음 실험을 진행한다.

- Linear Regression baseline
- Random Forest
- XGBoost / LightGBM 비교
- 군집 label 포함/제외 성능 비교
- 운영 보조 지표 포함/제외 비교
